# Lesson 12.2: Resume a case without reusing a stale approval

A case may change while an approval is waiting. The saved state records which version was reviewed. Follow the approval from the saved draft into the resume operation. Then compare it with an approval for a version that has been superseded.

[Companion lesson](../modules/12-stateful-agents/02-code-workshop.md) · [Original-source walkthrough](../modules/12-stateful-agents/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

The programme resumes one saved proposal and refuses an earlier approval after the proposal changes. Find the value that distinguishes those situations.

In [ ]:
import json

def prepare(case_id, revision):
    return {
        "case_id": case_id,
        "proposal_revision": revision,
        "stage": "awaiting_review",
        "proposal": {"action": "prepare_handoff", "source": "order-17"},
        "receipt": None,
    }

def resume(saved_state, approval, reviewer_cases):
    state = json.loads(saved_state)
    if approval["case_id"] not in reviewer_cases:
        raise PermissionError("Reviewer cannot act on this case")
    if approval["case_id"] != state["case_id"]:
        raise ValueError("Approval belongs to another case")
    if approval["proposal_revision"] != state["proposal_revision"]:
        raise ValueError("Approval belongs to an older proposal")
    if approval["decision"] != "approve":
        state["stage"] = "needs_revision"
        return state
    if state["stage"] == "completed" and state["receipt"]:
        return state
    if state["stage"] != "awaiting_review":
        raise ValueError("Case is not awaiting this review")
    state["receipt"] = {
        "status": "simulated",
        "action_id": f"{state['case_id']}:{state['proposal_revision']}",
    }
    state["stage"] = "completed"
    return state

state = prepare("C-17", 1)
snapshot = json.dumps(state)
approval = {"case_id": "C-17", "proposal_revision": 1, "decision": "approve"}
completed = resume(snapshot, approval, {"C-17"})
print(completed["stage"], completed["receipt"]["status"])

changed = prepare("C-17", 2)
try:
    resume(json.dumps(changed), approval, {"C-17"})
except ValueError as error:
    print(str(error))

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
assert completed["stage"] == "completed"
assert completed["receipt"]["status"] == "simulated"

## Try a change

Use the matching approval with an empty set of permitted cases. Does the proposal revision alone allow the action?

Make a prediction before opening the worked answer.

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

A matching revision tells the programme which proposal was reviewed. The access check answers a different question.

```python
try:
    resume(snapshot, approval, set())
except PermissionError as error:
    print(error)
else:
    raise AssertionError("The case should be outside scope")
```

</details>